# Wildlife Re-Identification (Problem 3) — Final Project Report
**Team NeuerXjensen:** Sabari Krishna R (`sabarikrishna2006`), Devyaansh (`devyaansh-9`), Jazl Anwar MV (`jazpeak`), Lokesh Akurathi (`Lokesh`)
**GitHub Repository:** `https://github.com/sabarikrishna2006/Animal_reid` (33/33 Kaggle submissions tagged & logged in `SUBMISSIONS.md`)
**Final Best Public LB:** **`0.88690` (Rank 1)** — `sub_team_blend_lnb39_swproto_g2q_mlp.csv` (`sub-v31-2026-11-27-blend-lnb39`)

---

### Executive Summary & Selected Three Model Versions (Evaluation Factor 2b)
This notebook contains the complete, rule-compliant codebase generating the Kaggle evaluation CSV files for our **three selected progressive model versions**, along with our Exploratory Data Analysis, error-driven design iterations, leak-free validation results, Private-LB anti-overfitting verification, limitations/capabilities analysis, and team individual contributions:

| Model Version | Git Tag & Reference Submission | Honest CV (MAP@5) | Public LB (MAP@5) | Gain vs. Previous | Total Gain vs. Baseline | Core Method |
|---|---|---:|---:|---:|---:|---|
| **Model 1 (Baseline)** | `sub-v02` (`sub_baseline_rawcos.csv`) | `0.8486` (CV v1) | **`0.76983`** | — | — | Raw L2 MiewID cosine + fixed per-species novelty threshold |
| **Model 2 (Intermediate)** | `sub-v05` (`sub_team_blend.csv`) | `0.8658` (shared CV) | **`0.85813`** | **`+0.08830`** | **`+0.08830`** | Gallery-only `PCA(256)->LDA` + pairwise RootSIFT RANSAC + LightGBM reranker + transductive propagation + 50/50 dual-pipeline blend |
| **Model 3 (Final Best)** | `sub-v31` (`sub_team_blend_lnb39_swproto_g2q_mlp.csv`) | **`0.88780`** (8-split shared) / **`0.93385`** (5-split Sabari) | **`0.88690`** | **`+0.02877`** | **`+0.11707`** | + `nprobe=64` LNBNN & `lnb_img` candidate expansion + 2-Stage Camera-Trap SIFT Killers + Full 2152-d $S_w^{-1/2}$ Whitening & Centroid Prototypes/DBA (`sw_proto`) + Transductive `G2Q` + non-tree Neural Net `MLPClassifier(128, 64)` + `sw_qq` learned `lprop` (`AUC 0.98265`) + Sabari `v26` |

**Strict Compliance Statement:** Uses **only** the provided MiewID (`2152`-d) embeddings, SIFT keypoints/descriptors, species, and viewpoint, and **only** permitted libraries (`numpy`, `pandas`, `scipy`, `scikit-learn`, `lightgbm`, `opencv-python` matching/RANSAC only, `faiss-cpu`, `networkx`, `matplotlib`). Zero raw images, zero deep-learning frameworks (`torch`/`tf`/`keras`/`jax`), zero `joblib`, zero external data, zero leakage.

In [1]:
import os, sys, json, time, pickle, subprocess
from pathlib import Path
import numpy as np
import pandas as pd

# Auto-detect repo root and competition data directory (works locally and on Kaggle)
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src" / "reid").exists():
    for cand in [REPO_ROOT.parent, Path("/kaggle/working/Animal_reid"), Path("teammate_sabari/repo")]:
        if (cand / "src" / "reid").exists():
            REPO_ROOT = cand.resolve()
            break

DATA_DIR = None
for cand in [
    REPO_ROOT / "data" / "reid_student_data_main",
    REPO_ROOT.parent.parent / "reid_student_data_main",
    Path("/kaggle/input/wildlife-reid/reid_student_data_main"),
    Path("/kaggle/input"),
]:
    if (cand / "train.csv").exists():
        DATA_DIR = cand.resolve()
        break

OUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else (REPO_ROOT / "outputs_final_3models")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Set RUN_FULL_TRAINING = True to retrain Model 2 & Model 3 from raw features (3-5 hrs on CPU).
# Leave False to run fast reproducible verification & CSV generation (< 10 sec) from saved pipeline artifacts.
RUN_FULL_TRAINING = False

print(f"REPO_ROOT : {REPO_ROOT}")
print(f"DATA_DIR  : {DATA_DIR}")
print(f"OUT_DIR   : {OUT_DIR}")

# 1. Run static rule-compliance scanner (allowed imports + allowed cv2 attributes only)
if (REPO_ROOT / "scripts" / "check_compliance.py").exists():
    res = subprocess.run([sys.executable, str(REPO_ROOT / "scripts" / "check_compliance.py")],
                         capture_output=True, text=True, check=True)
    print("Compliance check:", res.stdout.strip())

REPO_ROOT : C:\Users\Dev\Downloads\projectwild3\teammate_sabari\repo
DATA_DIR  : C:\Users\Dev\Downloads\projectwild3\reid_student_data_main
OUT_DIR   : C:\Users\Dev\Downloads\projectwild3\teammate_sabari\repo\outputs_final_3models
Compliance check: compliance OK


---
## 1. Exploratory Data Analysis & Leak-Free Open-Set Validation

### Key Dataset Findings (From `reports/eda.md` & Deep Error Analysis)
1. **Species Isolation:** `7,163` training images (`550` unique individuals) and `2,791` test images (`1,217` spotted hyenas, `1,574` leopards). Species labels are 100% clean, so all retrieval, metric learning, SIFT indexing, and transductive graphs run **strictly per species**.
2. **High Open-Set Novelty Rate:** Probing all-`new_individual` (`sub-v01`, `0.31113`) + EM prior estimation reveals ~`31–38%` of test queries belong to previously unseen individuals (`new_individual`), making calibrated open-set rejection critical.
3. **Why Random Splits Leak (Critical Discovery):** Early random splits kept burst photos of the same encounter across gallery and query, producing inflated CV (`0.91`) vs. Public LB (`0.80`). Adversarial validation (`AUC 0.83` on naive splits) showed test queries rarely have near-duplicate gallery siblings.
4. **Two Complementary Leak-Free CV Frameworks:**
   - **Dev 8-Split Honest CV (`per_enc=1`, `dev_pipeline/core.py`):** Clusters each identity into encounters (`cos > 0.80`), holds out `10%` of identities as `new_individual`, moves `40%` of remaining encounters to query, and thins to **1 query image per encounter**. Tracks Public LB within `~0.0009` (`0.88780` CV vs. `0.88690` LB).
   - **Sabari Official 5-Split CV v1 (`src/reid/splits.py`):** Average-linkage encounter clustering (`enc_thr=0.20`), species-stratified held-out `new_individual` (`15%` hyena, `40%` leopard), `p_move` (`30%` hyena, `25%` leopard), 5 seeds (`0..4`).
   - **Zero-Leakage Discipline:** Every transform (`PCA`, `LDA`, `Sw2152`, `sw_proto`, `r_g_diff`, `LNBNN`) fits **100% on fold galleries**; all stacking & `lprop` edge classifiers are strictly leave-one-split-out OOF (`0` query-ID overlap).

In [2]:
NEW = "new_individual"

tr = pd.read_csv(DATA_DIR / "train.csv")
te = pd.read_csv(DATA_DIR / "test.csv")
mids = pd.read_csv(DATA_DIR / "miewid_ids.csv")["id"].values
Xall = np.load(DATA_DIR / "miewid_features.npy").astype(np.float32)
pos = {k: i for i, k in enumerate(mids)}
Xtr = Xall[[pos[i] for i in tr["id"]]]
Xte = Xall[[pos[i] for i in te["id"]]]

def l2n(X):
    return X / np.maximum(np.linalg.norm(X, axis=1, keepdims=True), 1e-12)

allowed_by_sp = {sp: set(tr.loc[tr.species == sp, "identity"]) | {NEW} for sp in ["hyena", "leopard"]}
sp_te_map = dict(zip(te["id"], te["species"]))

def validate_sub_df(df, name):
    assert list(df.columns) == ["id", "predictions"], f"{name}: bad columns"
    assert len(df) == len(te) == 2791, f"{name}: bad row count {len(df)}"
    assert (df["id"].values == te["id"].values).all(), f"{name}: id order mismatch"
    assert df.isna().sum().sum() == 0, f"{name}: NaN detected"
    n_new_h, n_new_l, nh, nl = 0, 0, 0, 0
    for qid, pred_str in zip(df["id"].values, df["predictions"].values):
        labs = pred_str.split()
        sp = sp_te_map[qid]
        assert len(labs) == 5 and len(set(labs)) == 5, f"{name}: invalid top-5 on {qid}"
        assert set(labs) <= allowed_by_sp[sp], f"{name}: cross-species/invalid label on {qid}"
        if sp == "hyena":
            nh += 1; n_new_h += int(labs[0] == NEW)
        else:
            nl += 1; n_new_l += int(labs[0] == NEW)
    print(f"[VALIDATED] {name:38s} | rows={len(df)} | new@1: overall={(n_new_h+n_new_l)/len(df):.4f} (H={n_new_h/nh:.4f}, L={n_new_l/nl:.4f})")

print(f"Train: {len(tr)} rows, {tr.identity.nunique()} unique IDs | Test: {len(te)} rows | Overlap: {len(set(tr.id) & set(te.id))}")
for sp in ["hyena", "leopard"]:
    s_tr = tr[tr.species == sp]
    vc = s_tr.identity.value_counts()
    print(f"  {sp:7s}: train={len(s_tr):4d} ({len(vc):3d} IDs, 1-shot={(vc==1).sum():3d}, median={vc.median():.0f}) | test={(te.species==sp).sum():4d}")

Train: 7163 rows, 550 unique IDs | Test: 2791 rows | Overlap: 0
  hyena  : train=1912 (205 IDs, 1-shot=  3, median=6) | test=1217
  leopard: train=5251 (345 IDs, 1-shot= 99, median=3) | test=1574


---
## 2. Model Version 1 — Baseline (`sub-v02`, Public LB `0.76983`)

### Architecture
- **Representation:** Raw L2-normalized MiewID (`2152`-d) cosine similarity between each test query and every same-species training gallery image.
- **Identity Aggregation:** Maximum cosine similarity per gallery identity: $s(q, c) = \max_{g \in C_c} \langle u_q, u_g \rangle$.
- **Open-Set Decision:** Fixed per-species novelty threshold tuned on CV v1 (`tau = {hyena: 0.49, leopard: 0.52}`): insert `new_individual` at the first rank where $s(q, c) < \tau_{\text{sp}}$, and output the top 5 labels.

The cell below runs **100% live from raw MiewID features** and generates `model1_baseline_rawcos.csv`.

In [3]:
t0 = time.time()
Utr, Ute = l2n(Xtr), l2n(Xte)
TAU_BASE = {"hyena": 0.49, "leopard": 0.52}

preds_m1 = {}
for sp in ["hyena", "leopard"]:
    gm = (tr.species == sp).values
    qm = (te.species == sp).values
    glab = tr.identity.values[gm]
    qids = te.id.values[qm]
    S = Ute[qm] @ Utr[gm].T
    names, inv = np.unique(glab, return_inverse=True)
    I = np.column_stack([S[:, inv == j].max(1) for j in range(len(names))])
    tau = TAU_BASE[sp]
    for i, qid in enumerate(qids):
        order = np.argsort(-I[i])[:5]
        labs, placed = [], False
        for j in order:
            if not placed and I[i, j] < tau:
                labs.append(NEW); placed = True
            labs.append(names[j])
        if not placed:
            labs.append(NEW)
        preds_m1[qid] = " ".join(labs[:5])

sub_m1 = pd.DataFrame({"id": te["id"], "predictions": [preds_m1[q] for q in te["id"]]})
m1_path = OUT_DIR / "model1_baseline_rawcos.csv"
sub_m1.to_csv(m1_path, index=False)
validate_sub_df(sub_m1, "Model 1: Baseline RawCos (0.76983)")
print(f"Model 1 generated in {time.time() - t0:.2f}s -> {m1_path}")

[VALIDATED] Model 1: Baseline RawCos (0.76983)     | rows=2791 | new@1: overall=0.4493 (H=0.4248, L=0.4682)
Model 1 generated in 0.52s -> C:\Users\Dev\Downloads\projectwild3\teammate_sabari\repo\outputs_final_3models\model1_baseline_rawcos.csv


### Error Analysis Driving Model 1 (`0.76983`) → Model 2 (`0.85813`, `+0.08830`)
Inspecting Model 1's validation errors revealed four major bottlenecks:
1. **Within-individual nuisance variance:** Lighting, camera angle, and background clutter dominate raw MiewID cosine, compressing same-animal vs. impostor separation. → **Fix:** Gallery-only centering + `PCA(256) -> LDA(shrinkage=0.3)`.
2. **Global embeddings miss fine spot geometry:** Two different leopards/hyenas with similar pose score high in MiewID, while true matches with partial occlusion score low. → **Fix:** Pairwise **RootSIFT + Lowe ratio (`0.8`) + RANSAC homography & affine verification** (`cv2.findHomography`, `cv2.estimateAffinePartial2D`).
3. **Fixed threshold ignores query margin & density:** A single global $\tau$ cannot distinguish a blurry known animal from a sharp novel individual. → **Fix:** Supervised **LightGBM** `P(match | q, c)` and `P(new | q)` models + temperature-scaled softmax calibration.
4. **Test queries share unlabeled encounter structure:** Multiple photos of the same test individual reinforce each other. → **Fix:** Two-round neighbor stacking (`dev_pipeline`) + iterative transductive label propagation (`src/reid/`), combined in a **50/50 calibrated probability blend** (`sub-v05`, **`0.85813` Public LB**, **`+0.08830`** over baseline!).

---
## 3. Model Version 2 — Intermediate Dual-Pipeline Blend (`sub-v05`, Public LB `0.85813`)
- **Half 1 (50%, Sabari `v8`):** `configs/exp_v8_mix.json` — top-30 MiewID candidates, top-15 × 8-image RANSAC SIFT, multi-split LightGBM reranker + softmax calibration + iterative graph label propagation.
- **Half 2 (50%, Dev `v5`):** `dev_pipeline/model2.py` — top-20 `cos + PCA-LDA` candidates, top-5 × 20-image + ranks 6–10 × 2-image RANSAC SIFT, 2-round stacked LightGBM with query-query neighbor evidence (`K_QQ=5`), temperature-calibrated softmax.

In [4]:
m2_path = OUT_DIR / "model2_intermediate_team_blend.csv"

if RUN_FULL_TRAINING:
    # Full end-to-end training of Sabari v8_mix via official submission script
    subprocess.run([sys.executable, str(REPO_ROOT / "scripts" / "03_make_submission.py"),
                    "--config", str(REPO_ROOT / "configs" / "exp_v8_mix.json")], check=True)
    # Followed by 50/50 probability blend with Dev v5 probabilities via --blend
    print("Full training of Model 2 completed.")
else:
    # Fast reproducible verification from saved Model 2 (sub-v05 / v8_mix) artifacts
    m2_candidates = [
        REPO_ROOT.parent.parent / "solution" / "sub_team_blend.csv",
        REPO_ROOT / "submissions" / "sub_20261120-205512_sub_v8_mix.csv",
    ]
    src_m2 = next((p for p in m2_candidates if p.exists()), None)
    assert src_m2 is not None, "Model 2 reference CSV not found; set RUN_FULL_TRAINING=True to regenerate."
    sub_m2 = pd.read_csv(src_m2)
    sub_m2.to_csv(m2_path, index=False)
    validate_sub_df(sub_m2, "Model 2: Dual Blend v05 (0.85813)")
    print(f"Model 2 CSV ready -> {m2_path}")

[VALIDATED] Model 2: Dual Blend v05 (0.85813)      | rows=2791 | new@1: overall=0.3981 (H=0.3903, L=0.4041)
Model 2 CSV ready -> C:\Users\Dev\Downloads\projectwild3\teammate_sabari\repo\outputs_final_3models\model2_intermediate_team_blend.csv


### Error Analysis Driving Model 2 (`0.85813`) → Model 3 (`0.88690`, `+0.02877`, Total `+0.11707`)
Systematic error analysis across `sub-v06` through `sub-v30` uncovered **six orthogonal breakthroughs** that lifted Public LB from `0.85813` all the way to **Rank 1 `0.88690`**:

| Error Mode Diagnosed | Exact Zero-Leakage Solution Implemented | CV / LB Impact |
|---|---|---|
| **1. Missing MiewID candidates** (~`5%` of known animals missed MiewID top-20 despite matching spots) | Global **`nprobe=64` FAISS LNBNN** keypoint index (`lnb_img` top-5 candidate expansion + guided RANSAC SIFT + k-reciprocal re-ranking, `v10`) | **`0.85813 → 0.88073`** (`+0.02260`) |
| **2. Isolated-node over-dampening** during graph propagation | Degree-aware **`conn_new_scale`** + `lnb_uimg` / `lnb_rimg` background-keypoint masking (`v17`) | **`0.88073 → 0.88101`** |
| **3. Static camera-trap false SIFT matches** (trees, rocks, timestamps creating 0°-rotation RANSAC FPs) | **2-Stage Camera-Trap SIFT Killers** (Stage-1 corner `abs_rot < 0.50° & spread_min < 0.25`; Stage-2 full-frame `lnb_uimg / lnb_img < 0.35`) + gallery-only different-ID hubness **`r_g_diff`** + Missing-Candidate Veto Fix (`v19`, `bg3`) | **`0.88101 → 0.88375`** |
| **4. Noisy heuristic query-query propagation edges** | Strictly nested OOF **Learned BG-Filtered Query-Query Classifier (`lprop`, AUC `0.9777`)** (`lnb30`) | **`0.88375 → 0.88403`** |
| **5. `PCA(256)` discarded `1,896` of `2,152` MiewID dimensions** | **Full 2,152-d Within-Class Covariance Whitening ($S_w(\gamma)^{-1/2}$, $\gamma=0.5$, zero PCA truncation)** + `PCA(512)->LDA` + `Side-LDA` on both pipelines (`v23`, `v24`, `v25`, `dev_bg_sw2152`, `dev_v10_sw2152`) | **`0.88403 → 0.88585`** (`lnb34` `0.88424` → `lnb36` `0.88585`) |
| **6. Opposite-flank / 1-shot errors + false-hub query competition + 100%-LightGBM homogeneity** | **(a)** Gallery-only **`Sw2152` Centroid Prototypes + Same-ID Gallery DBA (`sw_proto`)** (`+0.0087` R@1, `+0.0179` Opposite-Side R@1; Sabari `v26` solo record `0.92586`); **(b)** Transductive **Gallery-to-Query Column Reciprocal Z/Share (`G2Q`)** (`v28`, `lnb38` `0.88605`); **(c)** Non-tree **Neural Network (`sklearn.neural_network.MLPClassifier(128, 64)`)** (beat LGB solo on BOTH Dev pools!); **(d)** **`sw_qq + ssw_qq` upgraded `lprop`** (`OOF AUC 0.9775 → 0.98265`) | **`0.88585 → 0.88690` (Rank 1, `lnb39`)** (8-split Honest CV `0.88571 → 0.88780`, DevHalf `0.89118`, Sabari 5-split `0.93385`) |

---
## 4. Model Version 3 — Final Best Team Blend (`sub-v31` `lnb39`, Public LB `0.88690` Rank 1)
- **Dev Half (`50%`, 2 independent candidate pools × 2 model families):**
  - `12.5%` `dev_bg_swproto_g2q_lgb_lprop` + `12.5%` `dev_bg_swproto_g2q_mlp_lprop` (`dev_pipeline/make_dev_bg_swproto_mlp.py`)
  - `12.5%` `dev_v10_swproto_g2q_lgb_lprop` + `12.5%` `dev_v10_swproto_g2q_mlp_lprop` (`dev_pipeline/make_dev_v10_swproto_mlp.py`)
- **Sabari Half (`50%`, 5 clean monotone-constrained LightGBM models, `alpha <= 0.8`):**
  - `10%` each of `exp_v17_bgclean_conn`, `exp_v19_bgclean_v2`, `exp_v24_fullbg_sw2152`, `exp_v25_fullbg_sw2152_lda512`, `exp_v26_fullbg_sw2152_proto`.
- **Final Decoding:** Simple arithmetic average of calibrated probability dictionaries → top 5 unique same-species labels (`dev_pipeline/make_team_blend.py --recipe lnb39`).

In [5]:
m3_path = OUT_DIR / "model3_best_lnb39_swproto_g2q_mlp.csv"

if RUN_FULL_TRAINING:
    # 1. Train Sabari 5-model monotone LGB ensemble
    for cfg_name in [
        "exp_v17_bgclean_conn.json", "exp_v19_bgclean_v2.json",
        "exp_v24_fullbg_sw2152.json", "exp_v25_fullbg_sw2152_lda512.json",
        "exp_v26_fullbg_sw2152_proto.json",
    ]:
        subprocess.run([sys.executable, str(REPO_ROOT / "scripts" / "03_make_submission.py"),
                        "--config", str(REPO_ROOT / "configs" / cfg_name)], check=True)
    # 2. Train Dev dual-pool LGB + MLP models (sw_proto + G2Q + sw_qq lprop)
    for sc in ["make_dev_bg_swproto_mlp.py", "make_dev_v10_swproto_mlp.py"]:
        subprocess.run([sys.executable, str(REPO_ROOT / "dev_pipeline" / sc)], check=True)
    # 3. Blend calibrated probabilities via make_team_blend.py --recipe lnb39
    subprocess.run([sys.executable, str(REPO_ROOT / "dev_pipeline" / "make_team_blend.py"),
                    "--recipe", "lnb39", "--out-dir", str(OUT_DIR)], check=True)
    sub_m3 = pd.read_csv(OUT_DIR / "sub_team_blend_lnb39_swproto_g2q_mlp.csv")
    sub_m3.to_csv(m3_path, index=False)
else:
    # Fast exact reproduction from saved calibrated .probs.pkl (if present) or verified repo CSV
    probs_pkl = REPO_ROOT.parent.parent / "solution" / "sub_team_blend_lnb39_swproto_g2q_mlp.probs.pkl"
    if probs_pkl.exists():
        blended = pickle.load(open(probs_pkl, "rb"))
        preds = []
        for q, sp in zip(te["id"].values, te["species"].values):
            sc = blended[q]
            ranked = sorted(sc, key=lambda l: (-sc[l], l))
            labs = [l for l in ranked if l in allowed_by_sp[sp]][:5]
            for l in [NEW] + sorted(allowed_by_sp[sp]):
                if len(labs) == 5: break
                if l not in labs: labs.append(l)
            preds.append(" ".join(labs))
        sub_m3 = pd.DataFrame({"id": te["id"], "predictions": preds})
        # Verify 100% byte-for-byte identity with submitted Rank 1 CSV
        ref_csv = REPO_ROOT / "submissions" / "sub_team_blend_lnb39_swproto_g2q_mlp.csv"
        if ref_csv.exists():
            assert sub_m3.equals(pd.read_csv(ref_csv)), "Mismatch vs submitted lnb39 CSV!"
            print("Exact 100% reproduction verified from sub_team_blend_lnb39_swproto_g2q_mlp.probs.pkl!")
    else:
        sub_m3 = pd.read_csv(REPO_ROOT / "submissions" / "sub_team_blend_lnb39_swproto_g2q_mlp.csv")
    sub_m3.to_csv(m3_path, index=False)

validate_sub_df(sub_m3, "Model 3: Best lnb39 Rank 1 (0.88690)")
print(f"Model 3 CSV ready -> {m3_path}")

Exact 100% reproduction verified from sub_team_blend_lnb39_swproto_g2q_mlp.probs.pkl!
[VALIDATED] Model 3: Best lnb39 Rank 1 (0.88690)   | rows=2791 | new@1: overall=0.3870 (H=0.3961, L=0.3799)
Model 3 CSV ready -> C:\Users\Dev\Downloads\projectwild3\teammate_sabari\repo\outputs_final_3models\model3_best_lnb39_swproto_g2q_mlp.csv


In [6]:
# Side-by-side comparison of the 3 generated Kaggle evaluation CSVs
m1_t1 = sub_m1["predictions"].str.split().str[0].values
m2_t1 = sub_m2["predictions"].str.split().str[0].values
m3_t1 = sub_m3["predictions"].str.split().str[0].values

summary = pd.DataFrame([
    {"Version": "Model 1 (Baseline)", "File": m1_path.name, "Public_LB": 0.76983, "Gain_vs_Base": "+0.00000",
     "new@1": round(float(np.mean(m1_t1 == NEW)), 4), "Top1_Agree_vs_M3": f"{100*np.mean(m1_t1==m3_t1):.2f}%"},
    {"Version": "Model 2 (Intermediate)", "File": m2_path.name, "Public_LB": 0.85813, "Gain_vs_Base": "+0.08830",
     "new@1": round(float(np.mean(m2_t1 == NEW)), 4), "Top1_Agree_vs_M3": f"{100*np.mean(m2_t1==m3_t1):.2f}%"},
    {"Version": "Model 3 (Best Final)", "File": m3_path.name, "Public_LB": 0.88690, "Gain_vs_Base": "+0.11707",
     "new@1": round(float(np.mean(m3_t1 == NEW)), 4), "Top1_Agree_vs_M3": "100.00%"},
])
print(summary.to_string(index=False))

               Version                                  File  Public_LB Gain_vs_Base  new@1 Top1_Agree_vs_M3
    Model 1 (Baseline)            model1_baseline_rawcos.csv    0.76983     +0.00000 0.4493           80.72%
Model 2 (Intermediate)    model2_intermediate_team_blend.csv    0.85813     +0.08830 0.3981           94.48%
  Model 3 (Best Final) model3_best_lnb39_swproto_g2q_mlp.csv    0.88690     +0.11707 0.3870          100.00%


---
## 5. Private-LB Anti-Overfitting & Zero-Leakage Verification

Every upgrade in Model 3 (`lnb39`, **`0.88690`**) was gated strictly on leak-free local CV before submission. Five independent stress tests confirm no overfitting and zero leakage:

1. **1,000× Monte Carlo 70% Private (`1,954`) / 30% Public (`837`) Simulation (Leak-Free 8-Split OOF):**
   - `lnb34` (`0.88424` LB): 70% Private sim = `0.88495 ± 0.00608`
   - `lnb36` (`0.88585` LB): 70% Private sim = `0.88560 ± 0.00607` (beats `lnb34` in `72.4%` of draws)
   - **`lnb39` (`0.88690` LB): 70% Private sim = `0.88752 ± 0.00605`** (beats `lnb36` in **`93.7%`** of all 1,000 Private 70% draws!)
2. **Held-Out Split-0 Learning Curves (Model-Level Overfitting):**
   - `dev_bg_swproto_g2q`: `Pair P(match)` val log-loss at iter `600` (`0.02207`) is within **`+0.00009`** of min (`0.02198`); `Query P(new)` at iter `400` (`0.17517`) within `+0.00135`.
   - `dev_v10_swproto_g2q`: `Pair P(match)` val log-loss at iter `600` (`0.01894`) is within **`+0.00012`** of min (`0.01882`); `MLPClassifier` uses `early_stopping=True` + L2 `alpha=0.05/0.10`; Sabari uses `GroupShuffleSplit` early stopping + monotonic constraints (`alpha <= 0.8`).
3. **Calibration Reliability (OOF ECE & Brier) & Test Stability:**
   - OOF Top-1 **ECE = `0.0493`**, **Brier = `0.0811`**, actual Top-1 accuracy **`84.92%`** > mean predicted confidence **`80.57%`** (zero overconfidence).
   - Test `mean P(new)` stayed identical (`0.3261` in `lnb36` → **`0.3254`** in `lnb39`).
   - Across 1,000 random 70% slices of `test.csv`: `new@1 = 0.3870 ± 0.0059`, `top1_conf = 0.8433 ± 0.0021`.
4. **Blend Perturbation & Leave-One-Out Stress Test (2,791 Test Queries):**
   - Shifting `Dev/Sab` to `45/55` or `55/45`: **`99.57–99.75%`** top-1 agreement.
   - Shifting `LGB/MLP` to `60/40` or `40/60`: **`99.93–100.00%`** top-1 agreement.
   - Dropping Sabari `v26` or `v17`: **`99.79–99.93%`** top-1 agreement.
5. **Monotonic CV ↔ Public LB Alignment:** `lnb34` (`0.88508` CV → `0.88424` LB) → `lnb36` (`0.88571` CV → `0.88585` LB) → **`lnb39` (`0.88780` CV → `0.88690` LB)**.

---
## 6. Capabilities & Limitations of the Best Model (Evaluation Factor 2c)
- **Capabilities:** Near-human accuracy on same-flank multi-shot individuals (`0.93+` MAP@5 on 6+ photo IDs); robust rejection of static camera-trap background false SIFT matches; well-calibrated open-set novelty detection; strong model-family diversity (LightGBM + Neural Net `MLPClassifier` + two independent candidate pools).
- **Limitations:**
  1. **Asymmetric opposite-flank queries** (left-side query vs. right-only gallery) remain the #1 error mode (`MAP 0.275` leopard / `0.488` hyena) because coat spots differ across flanks and raw images, timestamps, and camera-trap GPS coordinates are unavailable.
  2. **1-shot gallery identities** (`MAP 0.396` leopard / `0.552` hyena) suffer high viewpoint/lighting variance with only a single reference image.
  3. **Candidate-pool recall ceiling:** ~`3.5%` of known queries miss the top-30 MiewID/LNBNN candidate pool.

---
## 7. Individual Contributions (Team NeuerXjensen — Evaluation Factor 2d & Rule 3)

| Team Member | GitHub Account | Commits | Major Contributions |
|---|---|---:|---|
| **Devyaansh** | `devyaansh-9` | 12+ | Independent `dev_pipeline/` (2-round neighbor-stacked LightGBM, `wildlife_reid_solution.ipynb`), HotSpotter LNBNN (`src/reid/lnbnn.py`), keypoint query graph (`kpgraph.py`), `lnb_img` candidate/SIFT expansion & k-reciprocal re-ranking (`v9`/`v10`, `sub-v05..v11` up to **`0.88073`**); dual-pool `dev_bg_sw2152` + `exp_v25` + `make_team_blend.py` (`sub-v18..v24`, `sub-v29` **`0.88585`**); `Sw2152` centroid prototypes/DBA (`sw_proto`), non-tree Neural Net (`MLPClassifier`), `sw_qq` learned propagation (`sub-v31` Rank 1 **`0.88690`**), Private-LB anti-overfitting verification, and the final report notebook (`Team_NeuerXjensen_Final_Report.ipynb`). |
| **Jazl Anwar MV** | `jazpeak` / `Jazl` | 8+ | Read and researched animal re-ID papers by **Prof. Saket Anand** and related metric-learning literature, running experiments that guided team feature design; Stage-2 full-frame camera-trap SIFT filtering, gallery hubness (`r_g_diff`), missing-candidate veto fix & learned query-query graph propagation (`lprop`); gallery-only linear metric learning (`PCA(512)-LDA`, `Side-LDA`, and full 2152-dim `Sw` whitening `sw2152`, reaching Rank 1 **`0.88424`** in `sub-v26`); `v27` research-paper experiments (identity prototypes, gallery aggregation, and cross-flank alignment, `sub-v27`); `v28` gallery-to-query competitor features (`G2Q`) and safe `new_individual` gating (`sub-v30` Rank 1 **`0.88605`**, integrated into the final **`0.88690`** blend); blend ablations (`sub-v25`, `sub-v28`, `sub-v30a`), `dev_pipeline` helper modules (`core.py`, `model2.py`, `exp_blend_check.py`), and report documentation. |
| **Lokesh Akurathi** | `Lokesh-akurathi` | 7+ | Dual-pipeline `bg_clean` upgrades (`nprobe=64` LNBNN with `lnb_uimg`/`lnb_rimg` background-keypoint masking, Stage-1 corner camera-trap SIFT filter, and `conn_new_scale` isolated-node propagation fix driving `sub-v12..v17` up to **`0.88101`**); standalone Kaggle notebooks (`wildlife_reid_v4` `sub-v20`, `v25_v26_notebook.ipynb`), `v25` HNSW rescue + `v26` dual-model pipeline (`src/reid/dual_model.py`, `scripts/eval_blend.py`, `tests/test_v25_and_dual.py`), and experimental `alt_approach_v4.py` / `wildlife_reid_alt_approach.md`. |
| **Sabari Krishna R** | `sabarikrishna2006` | 3+ | Repository setup, compliance & leakage unit test suite (`tests/`), Exploratory Data Analysis (`eda/`, `reports/eda.md`), CV-vs-LB recalibration & error diagnostics (`scripts/diagnose_cvlb.py`, `error_breakdown.py`, `mix_cv.py`, `reports/gate1..5.md`), and the core `src/reid/` pipeline (encounter-disjoint open-set CV v1/v2, retrieval, SIFT RANSAC, monotone LightGBM reranker, softmax calibration, and transductive propagation powering `sub-v01..v04` and the 5-model Sabari half `v17..v26` of the final blend). |